In [2]:
# importing libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# reading the datasets

df_cat = pd.read_csv('data/category_tree.csv')
df_event = pd.read_csv('data/events.csv')
df_item_prop_1 = pd.read_csv('data/item_properties_part1.csv')
df_item_prop_2 = pd.read_csv('data/item_properties_part2.csv')

In [10]:
# combining the both item prop dfs
df_item_prop = pd.concat([df_item_prop_1, df_item_prop_2], ignore_index=True)

In [ ]:
del df_item_prop_1, df_item_prop_2

print(df_cat.shape, df_event.shape, df_item_prop.shape)

(1669, 2) (2756101, 5) (20275902, 4)


In [12]:
# basic stats

df_cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   categoryid  1669 non-null   int64  
 1   parentid    1644 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 26.2 KB


In [13]:
df_event.info()

<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          str    
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 105.1 MB


In [ ]:
#  logs form of data, contains the entry whenever anything is updated 

df_item_prop.info()

<class 'pandas.DataFrame'>
RangeIndex: 20275902 entries, 0 to 20275901
Data columns (total 4 columns):
 #   Column     Dtype
---  ------     -----
 0   timestamp  int64
 1   itemid     int64
 2   property   str  
 3   value      str  
dtypes: int64(2), str(2)
memory usage: 618.8 MB


In [26]:
# understanding what the table contains

df_cat.head()

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


In [27]:
df_event.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [28]:
df_item_prop.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


In [ ]:
# Checking for Nulls and Duplicates

print(df_cat.isna().sum())
print(df_cat.duplicated().sum())

categoryid     0
parentid      25
dtype: int64
0


In [19]:
print(df_event.isna().sum())
print(df_event.duplicated().sum())

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64
460


In [20]:
print(df_item_prop.isna().sum())
print(df_item_prop.duplicated().sum())

timestamp    0
itemid       0
property     0
value        0
dtype: int64
0


In [ ]:
# total unique items
df_cat.nunique()

categoryid    1669
parentid       362
dtype: int64

In [23]:
df_event[['visitorid', 'itemid', 'event', 'transactionid']].nunique()

visitorid        1407580
itemid            235061
event                  3
transactionid      17672
dtype: int64

In [24]:
# looking at the 3 unique events
df_event.event.unique()

<StringArray>
['view', 'addtocart', 'transaction']
Length: 3, dtype: str

In [ ]:
df_item_prop.nunique() # property has values - categoryid, available (0/1), rest all are hashed

timestamp         18
itemid        417053
property        1104
value        1966868
dtype: int64

In [34]:
# count of events
df_event.groupby('event')['itemid'].count()

event
addtocart        69332
transaction      22457
view           2664312
Name: itemid, dtype: int64

In [44]:
import duckdb

item_cat_map = duckdb.query("select itemid, value as cat_id, row_number() over(partition by itemid, property order by timestamp desc) rn from df_item_prop where property ='categoryid'").to_df()

In [47]:
print("unique items in item properties: ", df_item_prop.loc[df_item_prop['property']=='categoryid','itemid'].nunique())
item_cat_map[item_cat_map['rn']==1].count()

unique items in item properties:  417053


itemid    417053
cat_id    417053
rn        417053
dtype: int64

In [65]:
# lets get the top 5 items bought
item_orders = df_event.loc[df_event['event']=='transaction',['event','itemid']].groupby('itemid').count().sort_values(by='event', ascending=False).reset_index()

item_orders.head()

,itemid,event
0,461686,133
1,119736,97
2,213834,92
3,7943,46
4,312728,46


In [62]:
# top 5 categories

item_order_cnt_df = df_event.loc[df_event['event']=='transaction',['event','itemid']].groupby('itemid').count().sort_values(by='event', ascending=False).reset_index()

cat_orders = duckdb.query("select cat_id, sum(event) as orders from item_order_cnt_df a left join item_cat_map b on a.itemid = b.itemid group by 1 order by 2 desc").to_df()
cat_orders.head()

,cat_id,orders
0,1509,2015.0
1,1613,1994.0
2,491,1528.0
3,1120,900.0
4,720,724.0


In [73]:
# 380 items dont have category mapped to them

duckdb.query("select count(*) as items from item_order_cnt_df a left join item_cat_map b on a.itemid = b.itemid where b.itemid is null").to_df()

,items
0,380


In [68]:
# count of categories oc wise
cat_oc_q = """
select 
    case 
        when cum_oc < 0.1 then 'a.0to10'
        when cum_oc < 0.2 then 'b.10to20'
        when cum_oc < 0.3 then 'c.20to30'
        when cum_oc < 0.4 then 'd.30to40'
        when cum_oc < 0.5 then 'e.40to50'
        when cum_oc < 0.6 then 'f.50to60' 
        when cum_oc < 0.1 then 'g.60to70'
        when cum_oc < 0.1 then 'h.70to80'
        when cum_oc < 0.1 then 'i.80to90'
    else 'j.90to100' end as oc_decile,
    count(cat_id) as cat_count
from (
select 
    *,
    sum(orders) over(order by orders desc rows between unbounded preceding and current row)*1.0000/sum(orders) over() as cum_oc,
    sum(orders) over(order by orders desc rows between unbounded preceding and current row) as cum_orders, 
    sum(orders) over() as total_orders
from 
    cat_orders
)
group by 1
order by 1
"""

duckdb.query(cat_oc_q).to_df()

,oc_decile,cat_count
0,a.0to10,2
1,b.10to20,5
2,c.20to30,8
3,d.30to40,13
4,e.40to50,19
5,f.50to60,29
6,j.90to100,675


In [70]:
# count of items oc wise
item_oc_q = """
select 
    case 
        when cum_oc < 0.1 then 'a.0to10'
        when cum_oc < 0.2 then 'b.10to20'
        when cum_oc < 0.3 then 'c.20to30'
        when cum_oc < 0.4 then 'd.30to40'
        when cum_oc < 0.5 then 'e.40to50'
        when cum_oc < 0.6 then 'f.50to60' 
        when cum_oc < 0.1 then 'g.60to70'
        when cum_oc < 0.1 then 'h.70to80'
        when cum_oc < 0.1 then 'i.80to90'
    else 'j.90to100' end as oc_decile,
    count(itemid) as cat_count
from (
select 
    *,
    sum(event) over(order by event desc rows between unbounded preceding and current row)*1.0000/sum(event) over() as cum_oc,
    sum(event) over(order by event desc rows between unbounded preceding and current row) as cum_orders, 
    sum(event) over() as total_orders
from 
    item_orders

)
group by 1
order by 1
"""

duckdb.query(item_oc_q).to_df()

,oc_decile,cat_count
0,a.0to10,118
1,b.10to20,317
2,c.20to30,504
3,d.30to40,694
4,e.40to50,920
5,f.50to60,1123
6,j.90to100,8349


In [85]:
# customer segmentation

ordered_users = df_event[df_event.event == 'transaction'].visitorid.unique()
atc_users = df_event[df_event.event == 'transaction'].visitorid.unique()
viewed_users = df_event[df_event.event == 'view'].visitorid.unique()
all_users = df_event.visitorid.unique()
browsed_users = [u for u in all_users if u not in ordered_users]

print("num of ordered users: ", ordered_users.size)
print("num of atc users: ", atc_users.size)
print("num of viewed users: ", viewed_users.size)

print("num of all users: ", all_users.size)
print("num of all users who did not buy anything: ", len(browsed_users))


num of ordered users:  11719
num of atc users:  11719
num of viewed users:  1404179
num of all users:  1407580
num of all users who did not buy anything:  1395861


In [88]:
temp_array = np.isin(all_users, ordered_users)
temp_array[temp_array == False].size

1395861